A better version of `power_shendure_vs_minp.ipynb` with a more representative distribution of positive and negative effects. Specifically, we will be using the real values, plus many negatives.

Based on UKBB paper, expect ~30% of library to be active. So we will add 2x original size of negatives... 

We will also reduce computational burden by producing half-orthos.

# Imports & dask cluster creation

In [1]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster
from pathlib import Path

%load_ext autoreload
%autoreload 2

2026-01-23 16:00:35.320549: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-23 16:00:35.323548: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /apps/software/2024a/software/code-server/4.103.0/lib:/apps/software/2024a/software/gettext/0.22.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libiconv/1.17-GCCcore-13.3.0/lib:/apps/software/2024a/software/ncurses/6.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/apps/software/2024a/software/XZ/5.4.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/expat/2.6.2-GCCcore-13.3.0/lib:/apps/software/2024a/software/cUR

In [2]:
local=True
if local:
    cluster=LocalCluster(memory_limit='48G')
    client=Client(cluster)
else:
    cluster=SLURMCluster(
        cores=2,#cores per slurm job
        memory="32G",#memory per slurm job
        processes=1,#dask workers per slurm jobTrueT
        job_extra_directives=["-p day", 
            f"--job-name=simclust_worker",
            f"--time=4:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=2)
    client = Client(cluster,
            timeout=f"{5*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s"  # Worker heartbeat interval
        )

In [3]:
client.dashboard_link

'http://127.0.0.1:8787/status'

# Ground truth creation

We will use parameter estimates from by_cell_type models as we expect these to be the most accurate.

In [4]:
data_root=Path("/nfs/roberts/project/pi_skr2/shared/tabula_data")

In [5]:
primordial=scm.ortho.load(client,data_root/"shendure","ortho_primordial_v4")

In [6]:
primordial.compute_model_qc()

In [7]:
import pandas as pd

In [25]:
primordial.by_cell_type_parameters.nb

{'SurfaceEctoderm': <Future: finished, type: pandas.core.frame.DataFrame, key: lambda-0bfa44a7af1b466d0f7625a14da77794>,
 'NeuroectodermBrain': <Future: finished, type: pandas.core.frame.DataFrame, key: lambda-9a22b16141c2b7f906b23c6ac88b65d0>,
 'EpiblastPrimitiveStreak': <Future: finished, type: pandas.core.frame.DataFrame, key: lambda-a103a8e016e37bc034a1185a3ac51ca4>,
 'reference': <Future: finished, type: pandas.core.frame.DataFrame, key: lambda-a619bb9e0b7cd5e19f0c76b6d27443eb>,
 'ExEndodermVisceral': <Future: finished, type: pandas.core.frame.DataFrame, key: lambda-e8c2a7db1ba91341629fb742716fdeae>,
 'Mesoderm': <Future: finished, type: pandas.core.frame.DataFrame, key: lambda-825812cb5179d6d2a524ff6e12ae76d7>,
 'Cardiomyocytes': <Future: finished, type: pandas.core.frame.DataFrame, key: lambda-ac552f519bff8d666e1cb4a1e6ac91a7>,
 'ExEndodermParietal': <Future: finished, type: pandas.core.frame.DataFrame, key: lambda-f765c9d6cc58dd4153982e787c28c9c9>,
 'Haematoendothelial': <Futur

In [29]:
vals=[]
for key in primordial.by_cell_type_parameters.nb:
    working=primordial.by_cell_type_parameters.nb[key].result()
    print(len(working))
    

139
173
185
207
148
177
79
178
91
85


In [36]:
combos_in_original=primordial.training_data.data[["cre_id","cell_type"]].drop_duplicates()
combos_in_original

,cre_id,cell_type
0,Txndc12_chr4_7978,SurfaceEctoderm
1,Klf4_chr4_3952,SurfaceEctoderm
2,Foxa2_chr2_13840,SurfaceEctoderm
3,reference,SurfaceEctoderm
4,Lamc1_chr1_12152,SurfaceEctoderm
...,...,...
94287,Lamb1_chr12_2239,Cardiomyocytes
112928,Col1a2_chr6_93,ExEndodermParietal
154751,Col1a2_chr6_93,ExEndodermVisceral
197213,Cited2_chr10_1253,Cardiomyocytes


In [37]:
combos_in_original.groupby("cell_type").nunique()

,cre_id
cell_type,
Cardiomyocytes,79
EpiblastPrimitiveStreak,185
ExEndodermParietal,178
ExEndodermVisceral,148
Haematoendothelial,91
Mesoderm,177
NeuroectodermBrain,173
NeuroectodermRostral,85
SurfaceEctoderm,139


In [18]:
vals=[]
for key in primordial.by_cell_qc.keys():
    working=primordial.by_cell_qc[key]['dat'].reset_index().drop(columns=["mean(umis_mpra_bc)"])
    working["cell_type"]=key
    vals.append(working)
gt_cell_type=pd.concat(vals)

In [20]:
gt=pd.concat([gt_cre,gt_cell_type])
gt
gt.groupby(["cre_id","cell_type"]).mean().reset_index()

,cre_id,cell_type,mu
0,Bend5_chr4_8168,EpiblastPrimitiveStreak,0.040992
1,Bend5_chr4_8168,ExEndodermParietal,0.051741
2,Bend5_chr4_8168,NeuroectodermBrain,0.029262
3,Bend5_chr4_8168,SurfaceEctoderm,0.027477
4,Bend5_chr4_8168,reference,0.06772
...,...,...,...
1458,ubcP,Mesoderm,12.428503
1459,ubcP,NeuroectodermBrain,11.519773
1460,ubcP,NeuroectodermRostral,10.87317
1461,ubcP,SurfaceEctoderm,15.161613


In [22]:
# all expected cell types
all_cell_types = set(gt["cell_type"].unique())
n_cell_types = len(all_cell_types)

# count how many unique cell_types each cre_id has
counts = (
    gt.groupby("cre_id")["cell_type"]
      .nunique()
)

# cre_ids missing at least one cell_type
missing_cre_ids = counts[counts < n_cell_types].index.tolist()

len(missing_cre_ids)/len(gt["cre_id"].unique())

0.7451923076923077

In [ ]:
vals.dtypes

In [ ]:
vals["mu"] = vals["mu"].astype(float)

In [ ]:
vals.dtypes

Now that we have reasonable mu estimates for the real CRE, let us add 200% "indistinguishable from minP".

In [ ]:
#make "corresponding" inactive CREs...
mapping = {
    val: f"inactive_{i}"
    for i, val in enumerate(vals["cre_id"].unique())
}
mapping

In [ ]:
minP=scm.SHENDURE_BOUNDS.reference_activity
inactive=vals.copy().drop(columns=["mu"])
inactive["cre_id"] = inactive["cre_id"].map(mapping)
inactive["mu"]=minP
inactive

This is 100%. Let us double to 200%...

In [ ]:
# duplicated version with _b appended
inactive_b = inactive.copy()
inactive_b["cre_id"] = inactive_b["cre_id"] + "_b"

# stack them
inactive_double = pd.concat([inactive, inactive_b], ignore_index=True)
inactive_double

Then stack with original gt...

In [ ]:
final_gt=pd.concat([vals,inactive_double],ignore_index=True).rename({"mu":"true_mean"},axis=1)
final_gt

In [ ]:
assert len(final_gt[["cre_id","cell_type"]].drop_duplicates()) == len(final_gt)

# Creating artificial libraries

In [ ]:
libraries=[scm.simulate_library(CREs=final_gt["cre_id"],
                 library_model=scm.SHENDURE_BOUNDS.library_model)
                 for i in range(5)]

In [ ]:
libraries[2]

In [ ]:
final_gt.dtypes

# Creating sim

In [ ]:
sim=scm.de_novo_simulation(location=data_root,
                            name="twothird_pow_sim_2026-01-23",
                            client=client,
                            libraries=libraries,
                            library_mapping="corresponding",
                            n_sims=5,
                            experiment_bounds=scm.SHENDURE_BOUNDS,
                            ground_truth=final_gt)

In [ ]:
sim.gamut()

In [ ]:
#sim.save()

In [ ]:
#sim

Make the hypotheses...

In [ ]:
#spread_hypothesis.to_tsv(f"{data_root}/pow_sim_2026-01-03_hypo.tsv")
#hs_all_cre = scm.make_all_by_cre_hypotheses(
#    counts=demo_counts,
#    reference_cell_type="reference",
#)

In [ ]:
sim.ground_truth.dtypes

In [ ]:
client.close()
cluster.close()

In [ ]:
cells_df=scm.load_df_pickle_debug("permerge_cells_df_ba45c8b686c740f4b4aae36e46c280ab.pkl")
#cells_df=scm.cast_string_keys(cells_df,["cell_type", "cre_id"])
ground_truth=scm.load_df_pickle_debug("premerge_gt_0ee6c3cdca0a491299332a3d3e9ed250.pkl")
#ground_truth=scm.cast_string_keys(ground_truth,["cell_type", "cre_id"])

In [ ]:
cells_df.merge(ground_truth,
                on=["cell_type","cre_id"],
                validate="many_to_one",
                how="left",
                indicator=True)

In [ ]:
cells_df.sort_values(by=['cre_id', 'cell_type'])

In [ ]:
ground_truth.sort_values(by=['cre_id', 'cell_type'])

In [ ]:
cells_df[['cre_id', 'cell_type']].drop_duplicates()

In [ ]:
ground_truth[['cre_id', 'cell_type']].drop_duplicates()